# Member 6: Research Questions

**Member 6** - Interactive Reports, Deployment, and Testing

These research questions use the integrated dataset (`cd_total.csv` and `borough_month.csv`) produced by Member 3. They focus on the concentration of housing vs. crashes, and how total crash frequency correlates with new building filings across boroughs over time.

In [1]:
import pandas as pd
import plotly.express as px
from pathlib import Path

# Load the data
SEARCH_DIRS = [Path("../outputs/member3_integration"), Path("outputs/member3_integration")]
def find_file(file_name):
    for folder in SEARCH_DIRS:
        if (folder / file_name).exists():
            return folder / file_name
    raise FileNotFoundError(f"{file_name} not found.")

cd_total = pd.read_csv(find_file("cd_total.csv"))
borough_month = pd.read_csv(find_file("borough_month.csv"))

# --- Research Question 1 ---
# Do community districts with a higher share of the city's new housing units also experience a higher share of the city's serious traffic crashes?

total_city_units = cd_total["new_units"].sum()
total_city_ksi = cd_total["ksi"].sum()

cd_total["share_units"] = (cd_total["new_units"] / total_city_units) * 100
cd_total["share_ksi"] = (cd_total["ksi"] / total_city_ksi) * 100

fig1 = px.scatter(
    cd_total, 
    x="share_units", 
    y="share_ksi", 
    color="borough",
    hover_name="cd_label",
    size="new_units",
    labels={
        "share_units": "Share of NYC's proposed new housing units (%)",
        "share_ksi": "Share of NYC's serious crashes (KSI) (%)",
        "borough": "Borough"
    },
    title="RQ1: Concentration of New Housing vs. Concentration of Serious Crashes",
    template="plotly_white"
).add_hline(y=100/59, line_dash="dot", line_color="gray", annotation_text="Equal Distribution (1/59)").add_vline(x=100/59, line_dash="dot", line_color="gray")
fig1.show()

### RQ1 Interpretation
While some districts (like those in Queens and Brooklyn) have a very high share of the city's new housing, their share of serious crashes does not increase at the same rate. The data points are scattered, indicating that new housing development is highly concentrated in specific districts, but serious crashes are more evenly distributed across the city. This suggests that development alone does not dictate where severe traffic harm occurs.

In [3]:
# --- Research Question 2 ---
# In boroughs where the number of new building filings increased the most between 2022 and 2025, did the frequency of total crashes also increase proportionally?

# Filter for the full years 2022 and 2025
df_years = borough_month[borough_month["year"].isin([2022, 2025])]
borough_comp = df_years.groupby(["borough", "year"]).agg(
    new_filings=("new_filings", "sum"),
    crashes=("crashes", "sum")
).reset_index()

# Pivot to compare 2022 vs 2025
borough_pivot = borough_comp.pivot(index="borough", columns="year", values=["new_filings", "crashes"]).reset_index()

# Flatten the multi-level columns (this is what caused the error)
borough_pivot.columns = ["borough", "filings_2022", "filings_2025", "crashes_2022", "crashes_2025"]

# Calculate percentage changes
borough_pivot["filings_pct_change"] = ((borough_pivot["filings_2025"] - borough_pivot["filings_2022"]) / borough_pivot["filings_2022"]) * 100
borough_pivot["crash_pct_change"] = ((borough_pivot["crashes_2025"] - borough_pivot["crashes_2022"]) / borough_pivot["crashes_2022"]) * 100

fig2 = px.scatter(
    borough_pivot, 
    x="filings_pct_change", 
    y="crash_pct_change", 
    color="borough",
    hover_name="borough",
    text="borough",
    labels={
        "filings_pct_change": "Change in New Building Filings (2022 to 2025) %",
        "crash_pct_change": "Change in Total Crashes (2022 to 2025) %",
        "borough": "Borough"
    },
    title="RQ2: Change in Housing Filings vs. Change in Total Crashes (2022-2025)",
    template="plotly_white"
).add_hline(y=0, line_dash="dot", line_color="gray").add_vline(x=0, line_dash="dot", line_color="gray")
fig2.update_traces(textposition="top center")
fig2.show()

### RQ2 Interpretation
There is no clear correlation between the percentage change in new building filings and the percentage change in total crashes across the five boroughs. For example, some boroughs that saw a decrease in filings still experienced an increase in total crashes, and vice versa. This supports the broader project finding that housing development is not a direct driver of crash frequency.